# Gemma 2 LoRA: oncology research email drafting
Run in order on a GPU runtime. The two JSON datasets remain in your private Drive. Set `HF_TOKEN` in Colab Secrets and accept the Gemma model terms on Hugging Face first. This notebook saves an adapter only if training succeeds. Generated research claims must be checked by a person.

In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path('/content/gemma-oncology-lora')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/thinuka-rgb/gemma-oncology-lora.git', str(repo)], check=True)
os.chdir(repo)
print('Repository:', repo)

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login

token = userdata.get('HF_TOKEN')
if not token:
    raise RuntimeError('Add HF_TOKEN in Colab Secrets before continuing')
login(token=token)
del token
drive.mount('/content/drive')

In [ ]:
data_dir = Path('/content/drive/MyDrive/GemmaProject')
data_files = [data_dir / 'cold_email_dataset.json', data_dir / 'cold_email_dataset_1.json']
for path in data_files:
    if not path.is_file():
        raise FileNotFoundError(path)
subprocess.run(['python', 'train.py', '--validate-only', *map(str, data_files)], check=True)

In [ ]:
output_dir = data_dir / 'gemma-oncology-email-lora'
subprocess.run(['python', 'train.py', '--output', str(output_dir), *map(str, data_files)], check=True)
print('Saved:', output_dir / 'adapter')
print('Metrics:', output_dir / 'metrics.json')

In [ ]:
prompt = (
    'Draft a brief, polite email asking to learn about a research project. '
    'Only use these verified facts: I am a student interested in oncology; '
    'the researcher works on the project I describe below. '
    'Project details: [replace with verified details].'
)
subprocess.run(['python', 'generate.py', str(output_dir / 'adapter'), prompt], check=True)